Obtain ID's of records that have zero subjects.  
Then free to experiment with methods to enrich their metadata.

In [22]:
NO_RECORDS = 25

In [23]:
from datetime import datetime, timezone
from json import JSONDecodeError
from logging.config import dictConfig
from typing import Any, Optional
import psycopg
from psycopg import errors as psycopg_errors
from psycopg.rows import dict_row
from config.logging_config import LOGGING_CONFIG
from config.postgres_config import PostgresConfig
from tasks import transform_batch, add_file_metadata
import os
from fastapi import FastAPI, Query, HTTPException
import logging
from pydantic import BaseModel, Field
from utils.queue_utils import HarvestEventQueue, detect_identifier_type

from lxml import etree as ET
# API calls
import requests

In [24]:
dictConfig(LOGGING_CONFIG)
logger = logging.getLogger(__name__)

In [25]:
user = os.getenv("POSTGRES_USER")
passwd = os.getenv("POSTGRES_PASSWORD")
host = os.getenv("POSTGRES_ADDRESS", "127.0.0.1")

connection_params = {
    "user" : user,
    "password" : passwd,
    "dbname" : "datasetdb",
    "port" : 5432,
    "host" : host
}

In [27]:
query_records = """
            -- Exploring subjects from record raw metadata

-- Every xpath defines a potential XML structure with corresponding subject tag placement.
-- If raw_metadata conforms to ANY of these xpaths, subjects will be concatenated <subject> tags. (removing array_to_string() will return arrays)
-- If raw_metadata doesn't conform to any path (and probably has no subject tags), subjects will be ''


WITH query AS (
    SELECT
        records.*,
        array_to_string(
                xpath('oai:record/oai:metadata/datacite:resource/datacite:subjects/datacite:subject/text()', raw_metadata,
                    '{{datacite, http://datacite.org/schema/kernel-4},
                      {oai, http://www.openarchives.org/OAI/2.0/}}'
                )::text[] ||
				xpath('oai:record/oai:metadata/oaidc:oai_datacite/oaidc:payload/datacite:resource/datacite:subjects/datacite:subject/text()', raw_metadata,
                    '{{datacite, http://datacite.org/schema/kernel-4},
                      {oai, http://www.openarchives.org/OAI/2.0/},
                      {oaidc, http://schema.datacite.org/oai/oai-1.1/}}'
                )::text[] ||
				xpath('oai:record/oai:metadata/oai:resource/datacite:subjects/datacite:subject/text()', raw_metadata,
                    '{{datacite, http://datacite.org/schema/kernel-4},
                      {oai, http://www.openarchives.org/OAI/2.0/}}'
                )::text[] ||
				xpath('oai:record/oai:metadata/oad:oai_datacite/oad:payload/datacite:resource/datacite:subjects/datacite:subject/text()', raw_metadata,
				-- note the 3 in datacite
                    '{{datacite, http://datacite.org/schema/kernel-3},
					  {oai, http://www.openarchives.org/OAI/2.0/},
     				  {oad, http://schema.datacite.org/oai/oai-1.0/}}'
                )::text[] ||
				-- same path as previous but uses kernel-4 namespace
				xpath('oai:record/oai:metadata/oad:oai_datacite/oad:payload/datacite:resource/datacite:subjects/datacite:subject/text()', raw_metadata,
                    '{{datacite, http://datacite.org/schema/kernel-4},
					  {oai, http://www.openarchives.org/OAI/2.0/},
     				  {oad, http://schema.datacite.org/oai/oai-1.0/}}'
                )::text[]
				, ', '
        ) AS subjects
    FROM public.records
)
SELECT 
    id,
    title,
    raw_metadata,
    subjects
FROM query
WHERE query.subjects = ''
LIMIT %s
;
"""

In [28]:
try:
    with psycopg.connect(**connection_params, row_factory=dict_row) as conn:
        cur = conn.cursor()
        
        cur.execute(query_records, (NO_RECORDS,))
        
        results = cur.fetchall()
        print(results)

        record_ids = [str(row["id"]) for row in results]
        record_titles = [str(row["title"]) for row in  results]
        record_raw_metadata = [str(row["raw_metadata"]) for row in  results]


except Exception as e:
    logger.exception("Failed to fetch harvest runs")
    raise HTTPException(status_code=500, detail=str(e))

[{'id': '0725037e-553a-4a96-ac2b-a3f6a1d9f964::inv/87815817', 'title': 'Calibration cycle 2017-2', 'raw_metadata': '<record xmlns="http://www.openarchives.org/OAI/2.0/" xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance">\n  <header>\n    <identifier>oai:icat.isis.stfc.ac.uk:inv/87815817</identifier>\n    <datestamp>2019-01-18T09:15:05.475Z</datestamp>\n  </header>\n  <metadata>\n    <oai_datacite xmlns="http://schema.datacite.org/oai/oai-1.1/" xsi:schemaLocation="http://schema.datacite.org/oai/oai-1.1/ http://schema.datacite.org/oai/oai-1.1/oai.xsd">\n      <schemaVersion>4.3</schemaVersion>\n      <datacentreSymbol>BL.ISIS</datacentreSymbol>\n      <payload>\n        <resource xmlns="http://datacite.org/schema/kernel-4" xsi:schemaLocation="http://datacite.org/schema/kernel-4 http://schema.datacite.org/meta/kernel-4.3/metadata.xsd">\n          <identifier>https://data.isis.stfc.ac.uk/browse/instrument/5/facilityCycle//investigation/87815817</identifier>\n          <alternateIdentifi

In [29]:
record_titles

['Calibration cycle 2017-2',
 'Instrument Calibration',
 'Startup',
 'Vesuvio new backscattering setup calibration',
 'SANDALS interim instrument commissioning',
 'White Cryostat Commissioning',
 'Anvil gasket test',
 'Imperial Training Course (Day 2)',
 'HRPD cycle 19/2 calibration / detector commissioning',
 'Commissioning',
 'HRPD Calibration 2015/4',
 'Commissioning',
 'Calibration of the instrument',
 'NTC',
 'Investigation on spin dynamics of a staircase kagome material by using spin polarized muons',
 'Successive magnetic phase transition of the new frustrated compound, KCu3OCl(SO4)2',
 'New mica bank comissioning',
 'NTC2015',
 '2023 Larmor Calibration',
 'Offspec RF system commissioning',
 'ISIS003 statistical Approach',
 'ARGUS Commision',
 'TOSCA neutron spectrometer upon installation of the neutron guide (phase 1).',
 'IRIS Calibration',
 'Investigating the effect of framework structure and Si/Al ratio on the diffusion of branched alkanes in zeolite Y and ZSM-5']

In [30]:
record_raw_metadata[0]

'<record xmlns="http://www.openarchives.org/OAI/2.0/" xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance">\n  <header>\n    <identifier>oai:icat.isis.stfc.ac.uk:inv/87815817</identifier>\n    <datestamp>2019-01-18T09:15:05.475Z</datestamp>\n  </header>\n  <metadata>\n    <oai_datacite xmlns="http://schema.datacite.org/oai/oai-1.1/" xsi:schemaLocation="http://schema.datacite.org/oai/oai-1.1/ http://schema.datacite.org/oai/oai-1.1/oai.xsd">\n      <schemaVersion>4.3</schemaVersion>\n      <datacentreSymbol>BL.ISIS</datacentreSymbol>\n      <payload>\n        <resource xmlns="http://datacite.org/schema/kernel-4" xsi:schemaLocation="http://datacite.org/schema/kernel-4 http://schema.datacite.org/meta/kernel-4.3/metadata.xsd">\n          <identifier>https://data.isis.stfc.ac.uk/browse/instrument/5/facilityCycle//investigation/87815817</identifier>\n          <alternateIdentifiers alternateIdentifierType="RB">\n            <alternateIdentifier>1635080</alternateIdentifier>\n          </alte

In [31]:
def ET_from_string(string):
    return ET.ElementTree(ET.fromstring(string))

In [32]:
raw_xml = ET_from_string(record_raw_metadata[2])

In [33]:
raw_xml.getroot()

<Element {http://www.openarchives.org/OAI/2.0/}record at 0x273fd6da100>

In [34]:
print(ET.tostring(raw_xml, pretty_print=True).decode())

<record xmlns="http://www.openarchives.org/OAI/2.0/" xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance">
  <header>
    <identifier>oai:icat.isis.stfc.ac.uk:inv/61782241</identifier>
    <datestamp>2018-10-03T13:33:17.225Z</datestamp>
  </header>
  <metadata>
    <oai_datacite xmlns="http://schema.datacite.org/oai/oai-1.1/" xsi:schemaLocation="http://schema.datacite.org/oai/oai-1.1/ http://schema.datacite.org/oai/oai-1.1/oai.xsd">
      <schemaVersion>4.3</schemaVersion>
      <datacentreSymbol>BL.ISIS</datacentreSymbol>
      <payload>
        <resource xmlns="http://datacite.org/schema/kernel-4" xsi:schemaLocation="http://datacite.org/schema/kernel-4 http://schema.datacite.org/meta/kernel-4.3/metadata.xsd">
          <identifier>https://data.isis.stfc.ac.uk/browse/instrument/37/facilityCycle//investigation/61782241</identifier>
          <alternateIdentifiers alternateIdentifierType="RB">
            <alternateIdentifier>1535006</alternateIdentifier>
          </alternateIdentifie

In [35]:
# Finding a desired tag anywhere in tree, e.g. title
raw_xml.xpath("//*[local-name()='title']/text()")

['Startup']

In [36]:
element  = raw_xml.findall(".//{*}title")
print(element)



print(element[0].text)

[<Element {http://datacite.org/schema/kernel-4}title at 0x273fd6540c0>]
Startup


In [37]:
element  = raw_xml.findall(".//{*}identifier[@identifierType='DOI']")
element
#element[0].text

[]

In [ ]:
raw_xml.xpath("//*[local-name()='date']/text()")

['2015-06-01T23:00:00Z/2015-06-04T23:00:00Z', '2015-06-04T23:00:00Z']

In [39]:
# getting the DOI of the dataset
# Later, find related DOIs, articles...

# xpath returns a list of strings if /text() is included

raw_xml.xpath("//*[local-name()='identifier'][@identifierType='DOI']/text()")

[]